# Watch an Agent Do ML

**A multi-turn transcript: agent picks an algorithm, trains it, compares to alternatives, serves the winner.**

Every code cell below is a real TuiML tool call. The blockquotes show what an agent (Claude, ChatGPT, Gemini) would say at each step — the tool calls between them are exactly what the model emits when wired up via MCP or the provider's tool-use API.

If you have an agent connected ([Track A → Connect Your Agent](/tutorials/llm_friendly/02_mcp_server.ipynb)), you can paste each user prompt below and watch the agent reproduce these tool calls on its own.

## Setup: the tool harness

`execute_tool` is the same function the MCP server calls when an agent invokes a tool. We use it directly here so the cells run without an LLM in the loop, but the inputs and outputs are identical to what an agent sees.

In [ ]:
from tuiml.agent import execute_tool

## Turn 1 — User asks for help

> **User:** *"I have the iris dataset. Pick a good classifier for it and train one."*

**Agent's reasoning:** *"I'll search the algorithm catalog for a strong classifier baseline, then train it on iris with cross-validation."*

**Tool call 1.1:** `tuiml_list` — the catalog, filtered by keyword

In [ ]:
found = execute_tool("tuiml_list", category="algorithm", search="random forest")
for algo in found["components"][:3]:
    print(f"  {algo['name']:<48} {algo['type']}")

**Tool call 1.2:** `tuiml_train`

In [ ]:
result = execute_tool(
    "tuiml_train",
    algorithm="RandomForestClassifier",
    data="iris",
    target="class",
    cv=5,
)
print(f"Model ID: {result['model_id']}")
print(f"5-fold CV accuracy: {result['metrics']['cv_accuracy_score_mean']:.4f}")

**Agent's reply:** *"I trained a random forest with 5-fold CV. Mean accuracy is around 0.96. Want me to compare it against a few alternatives before we commit?"*

## Turn 2 — User asks to compare

> **User:** *"Yes, compare it to naive Bayes, SVM, and a decision tree. Whichever wins, save it."*

**Agent's reasoning:** *"One `tuiml_experiment` call covers the comparison. Then I'll retrain and save the winner."*

**Tool call 2.1:** `tuiml_experiment`

In [ ]:
exp = execute_tool(
    "tuiml_experiment",
    algorithms=[
        "RandomForestClassifier",
        "NaiveBayesClassifier",
        "KNearestNeighborsClassifier",
        "C45TreeClassifier",
    ],
    data="iris",
    target="class",
    cv=10,
)
print(exp["summary"])

**Agent's reply:** *"Here's the ranking — the top model has the best mean accuracy with the smallest spread. Saving the winner now."*

**Tool call 2.2:** `tuiml_save_model`

In [ ]:
scores = {
    algo: metrics["accuracy_score"]["mean"]
    for algo, metrics in exp["results"]["iris"].items()
}
best_algo = max(scores, key=scores.get)
print(f"Agent picked winner: {best_algo}  (accuracy {scores[best_algo]:.4f})")

winner = execute_tool("tuiml_train", algorithm=best_algo, data="iris", target="class")
saved = execute_tool(
    "tuiml_save_model",
    model_id=winner["model_id"],
    destination="agent_model.joblib",
)
print(f"Saved to: {saved.get('destination', 'agent_model.joblib')}")

## Turn 3 — User asks to deploy

> **User:** *"Great. Serve it on localhost so my app can hit it."*

**Agent's reasoning:** *"`tuiml_serve_model` starts a local REST endpoint and returns the URL. The saved file is the whole pipeline, so the endpoint takes raw feature rows."*

**Tool call 3.1:** `tuiml_serve_model` *(commented out so this notebook stays non-blocking — uncomment to actually launch a server in your own session)*

In [ ]:
# served = execute_tool("tuiml_serve_model", model_path="agent_model.joblib", port=8765)
# print(f"Endpoint: {served['url']}")
# print(f"Stop with: execute_tool('tuiml_stop_server', server_id=served['server_id'])")
print("Skipped: would start a REST server on :8765 in a real session.")
print("Agent reply: 'Endpoint live at http://localhost:8765/predict - POST features as JSON.'")

## What just happened

Three user turns. Five tool calls. No glue code, no scaffolding, no hallucinated parameter names. The agent navigated the full ML loop because every TuiML capability is a typed, schema-described tool.

**This is what the homepage means by *"agents can call it, discover it, trust it."***

### The same three turns in Python

Nothing here is agent-only. `tuiml.train()` returns a **fitted `Workflow`** — the pipeline *is* the model, so the save/serve steps are methods on the object the agent just produced:

```python
import tuiml

model = tuiml.train(
    {"name": "RandomForestClassifier"},
    {"source": "iris", "target": "class"},
    evaluation={"cv": 5},
)
print(model.metrics_)              # what tuiml_train returned as "metrics"

exp = tuiml.experiment(
    algorithms=["RandomForestClassifier", "NaiveBayesClassifier",
                "KNearestNeighborsClassifier", "C45TreeClassifier"],
    datasets=["iris"], cv=10,
)
print(exp.summary())

model.save("agent_model.joblib")   # not tuiml.save(...)
model.serve(port=8765)             # not tuiml.serve-only
```

### Try it for real

1. Connect an agent: [MCP setup](/tutorials/llm_friendly/02_mcp_server.ipynb) takes 5 minutes.
2. Paste the three user prompts above into your agent verbatim.
3. Watch it emit the same tool calls — possibly with smarter algorithm choices than ours.

### Where to go next

- **[Build an Agentic Workflow](/tutorials/llm_friendly/03_agentic_workflows.ipynb)** — turn this transcript into an autonomous loop with retries, error handling, and longer task horizons.
- **[Case Study: Diabetes Prediction](/tutorials/case_studies/01_diabetes_prediction.ipynb)** — same pattern on a real medical dataset, including SMOTE and serving.

In [ ]:
import os
if os.path.exists("agent_model.joblib"):
    os.remove("agent_model.joblib")